# OpsPilot — Streamlit Chat UI Launcher

This notebook starts the Streamlit chat interface as a background process and gives you the correct URL to open it inside Vocareum.

**Run cells in order (1 → 4). Cell 5 is optional for an embedded view. Cell 6 stops the server.**

| Cell | What it does |
|------|--------------|
| 1 | Install Streamlit |
| 2 | Find the project root |
| 3 | Start Streamlit in the background |
| 4 | Show clickable access URLs |
| 5 | Embed the UI directly in this notebook (optional) |
| 6 | Stop the Streamlit server |

In [ ]:
# Cell 1 — Install dependencies (including the Vocareum proxy bridge)
!pip install streamlit pysqlite3-binary jupyter-server-proxy -q
print('Dependencies ready ✓')
print()
print('NOTE: If jupyter-server-proxy was just installed for the first time,')
print('you may need to restart the Jupyter SERVER (not just the kernel).')
print('On Vocareum: File → Hub Control Panel → Stop My Server → Start My Server')

In [ ]:
# Cell 2 — Locate the project root and the app file
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())

# Walk up until we find streamlit_app.py
PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, Path('/voc/work')]:
    if (candidate / 'streamlit_app.py').exists():
        PROJECT_ROOT = candidate
        break

APP_PATH = PROJECT_ROOT / 'streamlit_app.py'

print(f'Project root : {PROJECT_ROOT}')
print(f'App path     : {APP_PATH}')
print(f'File exists  : {APP_PATH.exists()}')

if not APP_PATH.exists():
    print('\n❌  streamlit_app.py not found.')
    print('   Make sure you uploaded it to the project root (same folder as the agent/ directory).')
else:
    print('\n✅  Ready to launch.')

In [ ]:
# Cell 3 — Start Streamlit as a background process
import subprocess
import time

PORT = 8501

# Kill any previous instance on this port
os.system(f"fuser -k {PORT}/tcp 2>/dev/null || pkill -f 'streamlit run' 2>/dev/null || true")
time.sleep(1)

# Launch Streamlit with settings tuned for Vocareum / JupyterHub
proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
        f'--server.port={PORT}',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--browser.gatherUsageStats=false',
    ],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(f'Starting OpsPilot on port {PORT}...  (waiting 6 seconds)')
time.sleep(6)

if proc.poll() is None:
    print(f'✅  Streamlit is running!  PID = {proc.pid}')
    print('   → Run Cell 4 to get your access URL.')
else:
    # Read startup output to help diagnose the problem
    out = proc.stdout.read().decode('utf-8', errors='replace')
    print('❌  Streamlit failed to start. Output:')
    print(out[-2000:])

In [ ]:
# Cell 4 — Find and display the correct access URL
import os, socket, subprocess, re, json
from IPython.display import display, HTML

PORT = 8501

# ── 1. Server IP ──────────────────────────────────────────────────────────────
try:
    ext_ip = socket.gethostbyname(socket.gethostname())
except Exception:
    ext_ip = 'unknown'

# ── 2. Find which port the Jupyter server is running on ───────────────────────
jupyter_port = None

# Try modern jupyter server
try:
    r = subprocess.run(['jupyter', 'server', 'list', '--json'],
                       capture_output=True, text=True, timeout=5)
    for line in r.stdout.splitlines():
        try:
            m = re.search(r':(\d{4,5})/', json.loads(line).get('url', ''))
            if m:
                jupyter_port = int(m.group(1))
                break
        except Exception:
            continue
except Exception:
    pass

# Fallback: classic notebook list
if not jupyter_port:
    try:
        r = subprocess.run(['jupyter', 'notebook', 'list'],
                           capture_output=True, text=True, timeout=5)
        m = re.search(r'http[s]?://[^:]+:(\d+)', r.stdout)
        if m:
            jupyter_port = int(m.group(1))
    except Exception:
        pass

# Fallback: scan common Jupyter ports
if not jupyter_port:
    import socket as _s
    for p in [8888, 8889, 8080, 8000, 9000]:
        with _s.socket(_s.AF_INET, _s.SOCK_STREAM) as sock:
            if sock.connect_ex(('localhost', p)) == 0:
                jupyter_port = p
                break

# ── 3. Check jupyter-server-proxy ─────────────────────────────────────────────
sp = subprocess.run(['pip', 'show', 'jupyter-server-proxy'],
                    capture_output=True, text=True)
has_proxy = bool(sp.stdout.strip())

# ── 4. Check JupyterHub prefix (may be set in some Vocareum versions) ─────────
hub_prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')

# ── 5. Build URL options ───────────────────────────────────────────────────────
rows = ''

# Best: JupyterHub prefix (if set)
if hub_prefix:
    url = f'{hub_prefix}proxy/{PORT}/'
    rows += f'<tr style="background:#e8f5e9"><td><b>⭐ JupyterHub Proxy</b></td>' \
            f'<td><a href="{url}" target="_blank">{url}</a></td></tr>'

# Good: server-proxy via detected jupyter port
if has_proxy and jupyter_port:
    url = f'http://{ext_ip}:{jupyter_port}/proxy/{PORT}/'
    rows += f'<tr style="background:#e8f5e9"><td><b>⭐ Server Proxy (jupyter port {jupyter_port})</b>' \
            f'<br><small>Works if port {jupyter_port} is accessible from your browser</small></td>' \
            f'<td><a href="{url}" target="_blank">{url}</a></td></tr>'

# Fallback: direct external IP
rows += f'<tr><td><b>🌐 Direct IP</b><br><small>Only works if port {PORT} is open in the firewall</small></td>' \
        f'<td><a href="http://{ext_ip}:{PORT}" target="_blank">http://{ext_ip}:{PORT}</a></td></tr>'

# Manual guide
rows += f'''<tr style="background:#e3f2fd">
  <td><b>📋 Manual</b><br>
  <small>Look at your <b>current browser URL</b><br>e.g. http://1.2.3.4:8888/notebooks/...</small></td>
  <td>Copy the host:port from your browser URL<br>
  and add <code>/proxy/{PORT}/</code><br>
  e.g. <code>http://1.2.3.4:8888/proxy/{PORT}/</code></td></tr>'''

if not has_proxy:
    rows += f'''<tr style="background:#fff3e0">
      <td colspan="2">⚠️ <b>jupyter-server-proxy not active.</b>
      Run Cell 1 to install it, then restart the Jupyter server.<br>
      Until then, only the Direct IP option will work (if port {PORT} is open).</td></tr>'''

# ── 6. Print summary and display table ────────────────────────────────────────
print(f'Server IP          : {ext_ip}')
print(f'Jupyter port found : {jupyter_port or "not detected"}')
print(f'server-proxy       : {"✅ installed" if has_proxy else "❌ not installed — run Cell 1"}')
print(f'JupyterHub prefix  : {hub_prefix or "not set"}')

display(HTML(f'''
<div style="font-family:sans-serif;max-width:820px;margin-top:12px">
  <h3>🛡️ OpsPilot — Click a link to open the chat UI</h3>
  <table border="1" cellpadding="9" cellspacing="0"
         style="border-collapse:collapse;width:100%">
    <tr style="background:#1565c0;color:white">
      <th width="35%">Method</th><th>URL</th>
    </tr>
    {rows}
  </table>
  <p style="color:#666;font-size:.88em;margin-top:8px">
    ⭐ = most likely to work on Vocareum &nbsp;|&nbsp;
    Trailing <code>/</code> on proxy URLs is <b>required</b> — a blank page usually means it is missing.
  </p>
</div>
'''))

In [ ]:
# Cell 5 — Embed OpsPilot directly inside this notebook (optional)
# This works when the JupyterHub proxy URL is available (Cell 4 shows ⭐ link).
# If the iframe is blank, use the clickable URL from Cell 4 instead.

from IPython.display import IFrame, display

hub_prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')

if hub_prefix:
    proxy_url = f'{hub_prefix}proxy/{PORT}/'
    print(f'Embedding: {proxy_url}')
    display(IFrame(src=proxy_url, width='100%', height=750))
else:
    print('JupyterHub proxy prefix not found.')
    print(f'Open this URL manually in a new browser tab:')
    print(f'  http://localhost:{PORT}')

In [ ]:
# Cell 6 — Stop the Streamlit server
# Run this when you are done using the chat UI.

try:
    proc.terminate()
    proc.wait(timeout=5)
    print(f'✅  Streamlit (PID {proc.pid}) stopped.')
except Exception as e:
    print(f'Note: {e}')
    os.system("pkill -f 'streamlit run' 2>/dev/null || true")
    print('Sent kill signal to any streamlit process.')